In [2]:
# For this step we will use pyscenic on the data in order to execute the tutorial

<https://pyscenic.readthedocs.io/en/latest/tutorial.html>

<https://github.com/aertslab/SCENICprotocol/blob/master/notebooks/SCENIC%20Protocol%20-%20Case%20study%20-%20Mouse%20brain%20data%20set.ipynb>


In [2]:
# Define project-relative paths
PROJECT_DIR="$(pwd)"

SCENIC_DIR="${PROJECT_DIR}/data/scenic/totcd8"
RESOURCES_DIR="${PROJECT_DIR}/data/scenic/resources"
OUTPUT_DIR="${PROJECT_DIR}/results/scenic/totcd8"

This assumes the user runs the commands from the **root folder of the project**, for example:
```
project/
├── data/
│   └── scenic/
│       ├── totcd8/
│       │   ├── expression_matrix.tsv
│       │   └── filtered.loom
│       └── resources/
│           ├── TF_names_v_1.01.txt
│           ├── hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
│           ├── hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
│           └── motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl
└── results/
    └── scenic/
        └── totcd8/
            ├── expr_mat.adjacencies.tsv
            ├── regulons.csv
            └── final.loom
```

Before running pySCENIC, define the relative path to the folder containing the input files. 
This makes the workflow portable and avoids hardcoding user-specific absolute paths.


# Adjacency matrix
```bash
sudo docker run -it --rm \
    -v "${SCENIC_DIR}:/data" \
    -v "${RESOURCES_DIR}:/resources" \
    -v "${OUTPUT_DIR}:/output" \
    aertslab/pyscenic:0.12.1 pyscenic grn \
        --num_workers 25 \
        -o /output/expr_mat.adjacencies.tsv \
        /data/expression_matrix.tsv \
        /resources/TF_names_v_1.01.txt
```

# Regulons
```bash
sudo docker run -it --rm \
    -v "${SCENIC_DIR}:/data" \
    -v "${RESOURCES_DIR}:/resources" \
    -v "${OUTPUT_DIR}:/output" \
    aertslab/pyscenic:0.12.1 pyscenic ctx \
        /output/expr_mat.adjacencies.tsv \
        /resources/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
        /resources/hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
        --annotations_fname /resources/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl \
        --expression_mtx_fname /data/expression_matrix.tsv \
        --mode "custom_multiprocessing" \
        --output /output/regulons.csv \
        --num_workers 25
```

# AUCell
```bash
sudo docker run -it --rm \
    -v "${SCENIC_DIR}:/data" \
    -v "${OUTPUT_DIR}:/output" \
    aertslab/pyscenic:0.12.1 pyscenic aucell \
        /data/filtered.loom \
        /output/regulons.csv \
        -o /output/final.loom \
        --num_workers 5
```

So we end up having the regulons.csv as well as the expression adjacency matrix which contains our obtained regulons.
Let's load the data in here and provide the regulon analysis on inferenced GRNs from our data.